In [6]:
# Tugas 7
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [8]:
#A. EXTRACT
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


In [9]:
#B. TRANSFORM - Join & Kolom Turunan
df_gabungan_tugas7 = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                                 .join(df_menu, on="menu_id", how="inner")

df_gabungan_tugas7 = df_gabungan_tugas7.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan_tugas7.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [10]:
#C. TRANSFORM - Window Function
# Meringkas total penjualan per kombinasi cabang + menu terlebih dahulu
ringkasan_menu_cabang = df_gabungan_tugas7.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Window function untuk mengambil top-2 menu pada setiap cabang
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

[Stage 38:>                                                         (0 + 1) / 1]

+-------------------------+---------+---------------+---------+
|nama_cabang              |nama_menu|total_penjualan|peringkat|
+-------------------------+---------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |8160000        |2        |
+-------------------------+---------+---------------+---------+



In [11]:
#D. TRANSFORM - Spark SQL
df_gabungan_tugas7.createOrReplaceTempView("gabungan_tugas7")

penjualan_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan_tugas7
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [12]:
#E. LOAD
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

df_gabungan_tugas7.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.")
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

# Verifikasi: baca ulang lalu bandingkan jumlah baris
df_verifikasi = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris data gabungan (sebelum disimpan):", df_gabungan_tugas7.count())
print("Jumlah baris Parquet (setelah dibaca ulang)  :", df_verifikasi.count())

Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.
Found 4 items
-rw-r--r--   3 Azimah supergroup          0 2026-09-29 07:57 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - Azimah supergroup          0 2026-09-29 07:57 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - Azimah supergroup          0 2026-09-29 07:57 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - Azimah supergroup          0 2026-09-29 07:57 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


Jumlah baris data gabungan (sebelum disimpan): 4000
Jumlah baris Parquet (setelah dibaca ulang)  : 4000


In [13]:
#F. Ringkasan Eksekutif & Rekomendasi
ringkasan_eksekutif = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show(truncate=False)

[Stage 56:>                                                         (0 + 2) / 2]

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



In [14]:
from pyspark.sql.functions import (col, sum as spark_sum, avg, count, row_number, rank, dense_rank,
                                   lag, lead, when, broadcast, round as spark_round, countDistinct)
from pyspark.sql.window import Window

In [15]:
df_gabungan_tugas7.filter(col("qty") >= 3) \
    .groupBy("nama_cabang") \
    .agg(spark_sum("total_penjualan").alias("total")) \
    .explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[nama_cabang#129], functions=[sum(total_penjualan#206)])
   +- Exchange hashpartitioning(nama_cabang#129, 200), ENSURE_REQUIREMENTS, [plan_id=1687]
      +- HashAggregate(keys=[nama_cabang#129], functions=[partial_sum(total_penjualan#206)])
         +- Project [nama_cabang#129, (qty#113 * harga#154) AS total_penjualan#206]
            +- BroadcastHashJoin [menu_id#112], [menu_id#151], Inner, BuildRight, false
               :- Project [menu_id#112, qty#113, nama_cabang#129]
               :  +- BroadcastHashJoin [cast(cabang_id#111 as bigint)], [cabang_id#126L], Inner, BuildRight, false
               :     :- Filter (((isnotnull(qty#113) AND (qty#113 >= 3)) AND isnotnull(cabang_id#111)) AND isnotnull(menu_id#112))
               :     :  +- FileScan csv [cabang_id#111,menu_id#112,qty#113] Batched: false, DataFilters: [isnotnull(qty#113), (qty#113 >= 3), isnotnull(cabang_id#111), isnotnull(menu_id#112)], Form

In [16]:
df_gabungan_tugas7.cache()          # menandai untuk disimpan di memori
df_gabungan_tugas7.count()          # action pertama: menghitung DAN mengisi cache
df_gabungan_tugas7.count()          # action kedua: langsung ambil dari memori

print("Ter-cache:", df_gabungan_tugas7.is_cached)

df_gabungan_tugas7.unpersist()      # bebaskan memori setelah selesai

Ter-cache: True


DataFrame[menu_id: int, cabang_id: int, trx_id: string, qty: int, kepala_barista: string, kota: string, nama_cabang: string, nama_menu: string, kategori_menu: string, harga: int, total_penjualan: int]

In [18]:
df_broadcast = df_transaksi.join(broadcast(df_cabang), on="cabang_id") \
                           .join(broadcast(df_menu), on="menu_id")

df_broadcast.explain()
print("Jumlah baris:", df_broadcast.count())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [menu_id#112, cabang_id#111, trx_id#110, qty#113, kepala_barista#127, kota#128, nama_cabang#129, nama_menu#152, kategori_menu#153, harga#154]
   +- BroadcastHashJoin [menu_id#112], [menu_id#151], Inner, BuildRight, false
      :- Project [cabang_id#111, trx_id#110, menu_id#112, qty#113, kepala_barista#127, kota#128, nama_cabang#129]
      :  +- BroadcastHashJoin [cast(cabang_id#111 as bigint)], [cabang_id#126L], Inner, BuildRight, false
      :     :- Filter (isnotnull(cabang_id#111) AND isnotnull(menu_id#112))
      :     :  +- FileScan csv [trx_id#110,cabang_id#111,menu_id#112,qty#113] Batched: false, DataFilters: [isnotnull(cabang_id#111), isnotnull(menu_id#112)], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv], PartitionFilters: [], PushedFilters: [IsNotNull(cabang_id), IsNotNull(menu_id)], ReadSchema: struct<trx_id:string,cabang_id:int,menu_id:

In [19]:
data_contoh = [("A", 100), ("B", 90), ("C", 90), ("D", 80)]
df_contoh = spark.createDataFrame(data_contoh, ["menu", "skor"])

window_contoh = Window.orderBy(col("skor").desc())
df_contoh.withColumn("row_number", row_number().over(window_contoh)) \
         .withColumn("rank", rank().over(window_contoh)) \
         .withColumn("dense_rank", dense_rank().over(window_contoh)) \
         .show()

[Stage 76:=============================>                            (1 + 1) / 2]

+----+----+----------+----+----------+
|menu|skor|row_number|rank|dense_rank|
+----+----+----------+----+----------+
|   A| 100|         1|   1|         1|
|   B|  90|         2|   2|         2|
|   C|  90|         3|   2|         2|
|   D|  80|         4|   4|         3|
+----+----+----------+----+----------+



In [20]:
menu_dengan_selisih = ringkasan_menu_cabang \
    .withColumn("peringkat", row_number().over(window_cabang)) \
    .withColumn("selisih_dgn_atas", lag("total_penjualan", 1).over(window_cabang) - col("total_penjualan")) \
    .withColumn("menu_di_bawah", lead("nama_menu", 1).over(window_cabang))

menu_dengan_selisih.filter(col("nama_cabang") == "Kopi Nusantara Yogyakarta") \
                   .orderBy("peringkat").show(5, truncate=False)

[Stage 81:>                                                         (0 + 1) / 1]

+-------------------------+---------+---------------+---------+----------------+-------------+
|nama_cabang              |nama_menu|total_penjualan|peringkat|selisih_dgn_atas|menu_di_bawah|
+-------------------------+---------+---------------+---------+----------------+-------------+
|Kopi Nusantara Yogyakarta|Menu-12  |9275000        |1        |NULL            |Menu-15      |
|Kopi Nusantara Yogyakarta|Menu-15  |8160000        |2        |1115000         |Menu-9       |
|Kopi Nusantara Yogyakarta|Menu-9   |7175000        |3        |985000          |Menu-1       |
|Kopi Nusantara Yogyakarta|Menu-1   |7140000        |4        |35000           |Menu-3       |
|Kopi Nusantara Yogyakarta|Menu-3   |6390000        |5        |750000          |Menu-13      |
+-------------------------+---------+---------------+---------+----------------+-------------+
only showing top 5 rows



In [21]:
df_segmen = df_gabungan_tugas7.withColumn(
    "segmen_transaksi",
    when(col("total_penjualan") >= 100000, "Besar")
    .when(col("total_penjualan") >= 60000, "Sedang")
    .otherwise("Kecil")
)

df_segmen.groupBy("segmen_transaksi").agg(count("*").alias("jumlah_transaksi")) \
         .orderBy("segmen_transaksi").show()

+----------------+----------------+
|segmen_transaksi|jumlah_transaksi|
+----------------+----------------+
|           Besar|             736|
|           Kecil|            1741|
|          Sedang|            1523|
+----------------+----------------+



In [23]:
df_gabungan_tugas7.groupBy("nama_cabang") \
                  .pivot("kategori_menu") \
                  .agg(spark_sum("total_penjualan")) \
                  .show(truncate=False)

+-------------------------+--------+--------------+--------+
|nama_cabang              |Kopi    |Makanan Ringan|Non-Kopi|
+-------------------------+--------+--------------+--------+
|Kopi Nusantara Yogyakarta|21016000|45049000      |18580000|
|Kopi Nusantara Magelang  |21152000|48190000      |17796000|
|Kopi Nusantara Semarang  |22078000|49518000      |18946000|
+-------------------------+--------+--------------+--------+

